Setup source and target datasets.

In [0]:
ITEMS_SOURCE = "/Volumes/workspace/default/data_source/item.csv"
EVENTS_SOURCE = '/Volumes/workspace/default/data_source/event.csv'

spark.sql("""
          CREATE SCHEMA IF NOT EXISTS workspace.layer_1;
          """)

spark.sql("""
          CREATE SCHEMA IF NOT EXISTS workspace.layer_2;
          """)

spark.sql("""
          CREATE SCHEMA IF NOT EXISTS workspace.layer_3;
          """)

DataFrame[]

Read and save ITEM data source into layer_1, and process it for layer_2.

In [0]:
items = spark.read.option("header", "true") \
    .option("inferSchema", "false") \
    .csv(ITEMS_SOURCE)

items.write.saveAsTable('layer_1.items', mode='overwrite')

In [0]:
# spark.catalog.listCatalogs()
# spark.catalog.getDatabase('layer_1')

In [0]:
layer_1_items = spark.read.table('layer_1.items')

In [0]:
from pyspark.sql.functions import col, regexp_replace
# layer_1_items.withColumnsRenamed({
#     'adjective': 'item_adjective', 
#     'category': 'item_category', 
#     'created_at': 'item_created_at', 
#     'id': 'item_id', 
#     'modifier': 'item_modifier',
#     'name': 'item_name', 
#     'price': 'item_price'
# })
layer_2_items = layer_1_items.withColumnsRenamed({
    'adjective': 'item_adjective', 
    'category': 'item_category', 
    'id': 'item_id', 
    'modifier': 'item_modifier',
    'name': 'item_name', 
    
    # 'created_at': 'item_created_at', 
    # 'price': 'item_price'
}).withColumn('created_at', col('created_at').cast('date')).withColumnRenamed('created_at', 'item_created_at').withColumn('price', col('price').cast('numeric')).withColumn('item_id', regexp_replace("item_id", ".0$", ''))

layer_2_items.write.partitionBy('item_created_at').saveAsTable('layer_2.items', mode='overwrite')

Read and save EVENT data source into layer_1, and process it for layer_2.

In [0]:
events = spark.read.option("header", "true") \
    .option("inferSchema", "false") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .csv(EVENTS_SOURCE)

events.write.saveAsTable('layer_1.events', mode='overwrite')

Process layer 2 for events.

In [0]:
layer_2_events = spark.read.table('layer_1.events')

Process json column and clean up user id column.

In [0]:
from pyspark.sql.functions import col, from_json, regexp_replace, trim

json_schema = "event_name STRING, platform STRING, parameter_name STRING, parameter_value STRING"

events_with_parsed_payload = layer_2_events.withColumn(
    "parsed_payload", from_json(col("`event.payload`"), json_schema)
).withColumn('event_time', col('event_time').cast('date'))

Cast types for columns


In [0]:
from pyspark.sql.functions import col, expr, regexp_replace

final_events = events_with_parsed_payload.select(
  "event_id", "event_time", "user_id", "parsed_payload.*"
).withColumn('event_time', expr("try_cast(event_time AS TIMESTAMP)")).withColumn('user_id', regexp_replace('user_id', '.0$', ''))

final_events.write.partitionBy('event_time').option("overwriteSchema", "true").mode('overwrite').saveAsTable('layer_2.events')

Layer 3 datamart

In [0]:
items = spark.read.table('layer_2.items')
events = spark.read.table('layer_2.events')

In [0]:
from pyspark.sql.functions import year, count, desc, rank
from pyspark.sql.window import Window


view_events = events.filter(col("event_name") == "view_item").withColumn("view_year", year(col("event_time")))

final_report = view_events.join(
    items, 
    view_events["parameter_value"] == items["item_id"], 
    "inner"
)

result = final_report.groupby("item_name", "view_year") \
    .agg(count("event_id").alias("total_views")) \
    .orderBy("item_name", "view_year", "total_views", ascending=[True, True, False])

window = Window.partitionBy("view_year").orderBy(desc("total_views"))

result_with_rank = result.withColumn("rank", rank().over(window)) \
    .orderBy("item_name", "view_year", "rank")


In [0]:
from pyspark.sql.functions import row_number

platform_counts = final_report.groupby("item_name", "view_year", "platform") \
    .agg(count("event_id").alias("platform_view_count"))
platform_window = Window.partitionBy("item_name", "view_year").orderBy(desc("platform_view_count"))

most_used_platform_df = platform_counts.withColumn(
    "platform_rank", row_number().over(platform_window)
).filter(col("platform_rank") == 1) \
 .select("item_name", "view_year", col("platform").alias("most_used_platform"))

final_enhanced_report = result_with_rank.join(
    most_used_platform_df, 
    ["item_name", "view_year"], 
    "left"
)

final_enhanced_report.write.mode('overwrite').saveAsTable('layer_3.top_item')